In [1]:
import torch
import numpy as np
torch.manual_seed(42)
np.random.seed(42)

def sim_Data(T, N, m, p, f, with_covariates):
    Outcomes = []
    Thetas = []
    Lambdas = []
    latent_terms = []

    # Fixed base loadings
    Theta_base = torch.randn(m, p, dtype=torch.float64)
    Lambda_base = torch.randn(m, f, dtype=torch.float64)

    # step directions
    Theta_step = torch.randn(m, p, dtype=torch.float64)
    Lambda_step = torch.randn(m, f, dtype=torch.float64)

    for t in range(T):
        if t == 0:
            # start out at the base, before taking any steps
            Theta_t = Theta_base
            Lambda_t = Lambda_base
        else:
            # take a step, always in the same direction, but by varying amounts
            Theta_t = Theta_base + (Theta_step * torch.abs(torch.randn(1)))
            Lambda_t = Lambda_base + (Lambda_step * torch.abs(torch.randn(1)))
        # each time point append
        Thetas.append(Theta_t)
        Lambdas.append(Lambda_t)

    # if we aren't using covariates, just do the loadings for the latent factors
    Z_F = torch.randn(p, N, dtype=torch.float64) * bool(with_covariates)  # (p x N)
    M_F= torch.randn(f, N, dtype=torch.float64)  # (f x N)

    for t in range(T):
        # returning the latent terms
        L_t = Lambdas[t] @ M_F
        latent_terms.append(L_t)

        Y_t = Thetas[t] @ Z_F + Lambdas[t] @ M_F
        Outcomes.append(Y_t)

    return Outcomes, latent_terms, M_F, Z_F




In [2]:


import torch
import numpy as np
torch.manual_seed(42)
np.random.seed(42)
from learnQorthogonal import learnQ

# settings
T = 40
N = 50
m = 10
f = 3
# since we aren't passing covariates, p doesn't matter here.
p = 2

with_covariates = False

Outcomes, latent_terms, M_F,_ = sim_Data(T, N, m, p, f, with_covariates)

targets = []
covariates = []
for outcome in Outcomes:
    # take the first column
    targets.append(outcome[:, 0])
    # all but the first column
    covariates.append(outcome[:, 1:])

print(covariates[-1].shape)

Q_final, w_final = learnQ(targets, covariates, embedding_dim=3, n_iterations=1000, reg_Q=1, reg_w=100, verbose=False, num_timepoints = None, init_Q = "id")

print("################################")
print("covariates[-1].shape:", covariates[-1].shape)
print("Q_final shape:", Q_final.shape)
print("w_final shape:", w_final.shape)
print("synthetic covariates: \n", covariates[-1] @ Q_final @ w_final)

(CVXPY) Apr 22 08:29:20 PM: Encountered unexpected exception importing solver DIFFCP:
ImportError('diffcp >= 1.0.15 is required')
torch.Size([10, 49])
################################
covariates[-1].shape: torch.Size([10, 49])
Q_final shape: (49, 3)
w_final shape: (3,)
synthetic covariates: 
 tensor([-2.58, -0.97,  3.31,  0.60,  2.73, -2.57, -1.81,  1.08,  2.07, -2.36],
       dtype=torch.float64)


Need to see what Monte Carlo Bias looks like

In [4]:
pred_Q = covariates[-1] @ Q_final @ w_final
print(pred_Q)
bias = (pred_Q - targets[-1])
print(torch.norm(bias))

nMinus1, D = Q_final.shape
minDim = min(nMinus1, D)
print(np.linalg.norm(Q_final.T @ Q_final - np.eye(minDim)))

tensor([-2.58, -0.97,  3.31,  0.60,  2.73, -2.57, -1.81,  1.08,  2.07, -2.36],
       dtype=torch.float64)
tensor(0.02, dtype=torch.float64)
7.5293092315519525e-16


## Initializing with different matrices seems to give same solution for prediction

In [5]:
# Run with two different initializations
Q1, w1 = learnQ(targets, covariates, embedding_dim=3, n_iterations=1000, reg_Q=1, reg_w=100, verbose=False, init_Q = "id")
Q2, w2 = learnQ(targets, covariates, embedding_dim=3, n_iterations=1000, reg_Q=1, reg_w=100, verbose=False, init_Q = "random")

# do predictions agree?
pred1 = covariates[-1] @ Q1 @ w1
pred2 = covariates[-1] @ Q2 @ w2

In [6]:
print(pred1)
print(pred2)

tensor([-2.58, -0.97,  3.31,  0.60,  2.73, -2.57, -1.81,  1.08,  2.07, -2.36],
       dtype=torch.float64)
tensor([-2.58, -0.97,  3.31,  0.61,  2.72, -2.58, -1.81,  1.08,  2.08, -2.36],
       dtype=torch.float64)
